#Question -1(a) Image Classification on MNIST and FashionMNIST Dataset, using Resnet18 and Resnet50

Importing Necessary Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import numpy as np
import random

Setting up seed and Hyperparams grid

In [2]:
# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---- Hyperparameters ----
BATCH_SIZES = [16, 32]
LEARNING_RATES = [1e-3, 1e-4]
OPTIMIZERS = ["SGD", "Adam"]
EPOCHS_LIST = [1,2]
PIN_MEMORY_OPTIONS = [False, True]

USE_AMP = True  # Constant as mentioned


Loading dataset and performing train-val-test split (70-10-20)

In [3]:
# ---- Dataset Loader ----
def get_dataloaders(dataset_name, batch_size, pin_memory):
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == "MNIST":
        dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)
    else:
        dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
        test_dataset = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

    total_len = len(dataset)
    train_len = int(0.7 * total_len)
    val_len = int(0.1 * total_len)
    rem_len = total_len - train_len - val_len

    train_set, val_set, _ = random_split(dataset, [train_len, val_len, rem_len])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=pin_memory)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=pin_memory)

    return train_loader, val_loader, test_loader

Loading Resnet18 & Resnet50 Models, along with optimizers

In [4]:
# ---- Model Builder ----
def get_model(model_name):
    if model_name == "resnet18":
        model = models.resnet18(weights=None)
    else:
        model = models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

# ---- Optimizer Builder ----
def get_optimizer(name, params, lr):
    if name == "SGD":
        return optim.SGD(params, lr=lr, momentum=0.9)
    return optim.Adam(params, lr=lr)

Train function

In [5]:
# ---- Train & Evaluate ----
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with autocast("cuda", enabled=USE_AMP):
            outputs = model(x)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, preds = outputs.max(1)
        correct += preds.eq(y).sum().item()
        total += y.size(0)

    return 100.0 * correct / total

Evaluation Function

In [6]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, preds = outputs.max(1)
            correct += preds.eq(y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total

Main Loop to train and evaluate for the grid of hyperparams

In [7]:
# ---- Experiment Loop ----
for dataset_name in ["MNIST", "FashionMNIST"]:
    for model_name in ["resnet18", "resnet50"]:
        for batch_size in BATCH_SIZES:
            for lr in LEARNING_RATES:
                for opt_name in OPTIMIZERS:
                    for epochs in EPOCHS_LIST:
                        for pin_memory in PIN_MEMORY_OPTIONS:

                            print("\n" + "=" * 90)
                            print(f"DATASET     : {dataset_name}")
                            print(f"MODEL       : {model_name}")
                            print(f"BATCH SIZE  : {batch_size}")
                            print(f"LR          : {lr}")
                            print(f"OPTIMIZER  : {opt_name}")
                            print(f"EPOCHS     : {epochs}")
                            print(f"PIN_MEMORY : {pin_memory}")
                            print("=" * 90)

                            train_loader, val_loader, test_loader = get_dataloaders(
                                dataset_name, batch_size, pin_memory
                            )

                            model = get_model(model_name)
                            optimizer = get_optimizer(opt_name, model.parameters(), lr)
                            criterion = nn.CrossEntropyLoss()
                            scaler = GradScaler("cuda", enabled=USE_AMP)

                            for ep in range(epochs):
                                train_acc = train_one_epoch(
                                    model, train_loader, optimizer, criterion, scaler
                                )
                                val_acc = evaluate(model, val_loader)

                                print(f"Epoch [{ep+1}/{epochs}] | "
                                      f"Train Acc: {train_acc:.2f}% | "
                                      f"Val Acc: {val_acc:.2f}%")

                            test_acc = evaluate(model, test_loader)
                            print(f"FINAL TEST ACCURACY: {test_acc:.2f}%")


DATASET     : MNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 1
PIN_MEMORY : False


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 341kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.78MB/s]


Epoch [1/1] | Train Acc: 90.22% | Val Acc: 97.87%
FINAL TEST ACCURACY: 98.06%

DATASET     : MNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 1
PIN_MEMORY : True
Epoch [1/1] | Train Acc: 89.85% | Val Acc: 97.32%
FINAL TEST ACCURACY: 97.64%

DATASET     : MNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 2
PIN_MEMORY : False
Epoch [1/2] | Train Acc: 89.83% | Val Acc: 97.98%
Epoch [2/2] | Train Acc: 98.09% | Val Acc: 98.55%
FINAL TEST ACCURACY: 99.04%

DATASET     : MNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 2
PIN_MEMORY : True
Epoch [1/2] | Train Acc: 89.70% | Val Acc: 97.58%
Epoch [2/2] | Train Acc: 98.09% | Val Acc: 98.80%
FINAL TEST ACCURACY: 98.70%

DATASET     : MNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : Adam
EPOCHS     : 1
PIN_MEMORY : False
Epoch [1/1] | Train Acc: 95.74% | Val Acc: 98.23%
FINAL TEST 

100%|██████████| 26.4M/26.4M [00:00<00:00, 113MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.96MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 62.5MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.1MB/s]


Epoch [1/1] | Train Acc: 78.13% | Val Acc: 86.88%
FINAL TEST ACCURACY: 85.96%

DATASET     : FashionMNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 1
PIN_MEMORY : True
Epoch [1/1] | Train Acc: 78.08% | Val Acc: 83.75%
FINAL TEST ACCURACY: 83.69%

DATASET     : FashionMNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 2
PIN_MEMORY : False
Epoch [1/2] | Train Acc: 77.82% | Val Acc: 86.97%
Epoch [2/2] | Train Acc: 87.90% | Val Acc: 89.30%
FINAL TEST ACCURACY: 88.44%

DATASET     : FashionMNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : SGD
EPOCHS     : 2
PIN_MEMORY : True
Epoch [1/2] | Train Acc: 78.07% | Val Acc: 86.32%
Epoch [2/2] | Train Acc: 87.85% | Val Acc: 88.32%
FINAL TEST ACCURACY: 87.20%

DATASET     : FashionMNIST
MODEL       : resnet18
BATCH SIZE  : 16
LR          : 0.001
OPTIMIZER  : Adam
EPOCHS     : 1
PIN_MEMORY : False
Epoch [1/1] | Train Acc: 81.83% |

#Question -1(b) Training and Testing SVM with various hyperparameters on MNIST and FashionMNIST

Importing Necessary Libraries

In [8]:
import time
import numpy as np
import random
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Setting up seed and hyperparameter grid, also perform train-val-test split (70-10-20)

In [9]:
# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
seed_everything()

# ---- Hyperparameters ----
KERNELS = ["rbf", "poly"]
C_VALUES = [0.1, 1, 10]
GAMMA_VALUES = ["scale", "auto"]
DEGREES = [2, 3]
MAX_SAMPLES = 20000

# ---- Dataset Loader (Flattened for SVM) ----
def load_dataset(dataset_name):
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    if dataset_name == "MNIST":
        dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
    else:
        dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)

    X = dataset.data.numpy().reshape(len(dataset), -1) / 255.0
    y = dataset.targets.numpy()

    if len(X) > MAX_SAMPLES:
        idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
        X, y = X[idx], y[idx]

    return X, y

Main Experiment loop training and testing for all hyperparam values

In [10]:
# ---- Experiment Loop ----
for dataset_name in ["MNIST", "FashionMNIST"]:
    print("\n" + "=" * 100)
    print(f"DATASET: {dataset_name}")
    print("=" * 100)

    X, y = load_dataset(dataset_name)

    # 70-10-20 split
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=2/3, random_state=42, stratify=y_temp
    )

    for kernel in KERNELS:
        for C in C_VALUES:
            for gamma in GAMMA_VALUES:

                if kernel == "poly":
                    for degree in DEGREES:
                        print("\n" + "-" * 90)
                        print(f"Kernel={kernel} | C={C} | gamma={gamma} | degree={degree}")

                        svm = SVC(
                            kernel=kernel,
                            C=C,
                            gamma=gamma,
                            degree=degree
                        )

                        start_time = time.time()
                        svm.fit(X_train, y_train)
                        end_time = time.time()

                        train_time_ms = (end_time - start_time) * 1000
                        y_pred = svm.predict(X_test)
                        test_acc = accuracy_score(y_test, y_pred) * 100

                        print(f"Training Time (ms): {train_time_ms:.2f}")
                        print(f"Test Accuracy (%): {test_acc:.2f}")

                else:  # rbf kernel
                    print("\n" + "-" * 90)
                    print(f"Kernel={kernel} | C={C} | gamma={gamma}")

                    svm = SVC(
                        kernel=kernel,
                        C=C,
                        gamma=gamma
                    )

                    start_time = time.time()
                    svm.fit(X_train, y_train)
                    end_time = time.time()

                    train_time_ms = (end_time - start_time) * 1000
                    y_pred = svm.predict(X_test)
                    test_acc = accuracy_score(y_test, y_pred) * 100

                    print(f"Training Time (ms): {train_time_ms:.2f}")
                    print(f"Test Accuracy (%): {test_acc:.2f}")


DATASET: MNIST

------------------------------------------------------------------------------------------
Kernel=rbf | C=0.1 | gamma=scale
Training Time (ms): 28369.39
Test Accuracy (%): 92.95

------------------------------------------------------------------------------------------
Kernel=rbf | C=0.1 | gamma=auto
Training Time (ms): 51821.00
Test Accuracy (%): 86.83

------------------------------------------------------------------------------------------
Kernel=rbf | C=1 | gamma=scale
Training Time (ms): 13599.59
Test Accuracy (%): 96.05

------------------------------------------------------------------------------------------
Kernel=rbf | C=1 | gamma=auto
Training Time (ms): 19822.22
Test Accuracy (%): 91.65

------------------------------------------------------------------------------------------
Kernel=rbf | C=10 | gamma=scale
Training Time (ms): 12867.16
Test Accuracy (%): 96.70

------------------------------------------------------------------------------------------
Kern